In [ ]:
import numpy as np
import pygame
import sys
import math

# -----------------------------
# Connect-4 (Pygame GUI)
# -----------------------------
total_row = 6
total_coloum = 7

BLUE = (0, 0, 255)
BLACK = (0, 0, 0)
RED = (255, 0, 0)
YELLOW = (255, 255, 0)

def create_board():
    return np.zeros((total_row, total_coloum), dtype=int)

def drop_piece(board, row, coloum, piece):
    board[row][coloum] = piece

def is_vlid_location(board, coloum):
    # row 0 is bottom in our board logic, so the "top" is total_row-1
    return board[total_row - 1][coloum] == 0

def get_next_open_row(board, coloum):
    for row in range(total_row):
        if board[row][coloum] == 0:
            return row
    return None

def print_board(board):
    print(np.flip(board, 0))

def winning_move(board, piece):
    # horizontal
    for coloum in range(total_coloum - 3):
        for row in range(total_row):
            if (board[row][coloum] == piece and
                board[row][coloum + 1] == piece and
                board[row][coloum + 2] == piece and
                board[row][coloum + 3] == piece):
                return True

    # vertical
    for coloum in range(total_coloum):
        for row in range(total_row - 3):
            if (board[row][coloum] == piece and
                board[row + 1][coloum] == piece and
                board[row + 2][coloum] == piece and
                board[row + 3][coloum] == piece):
                return True

    # diagonal positive
    for coloum in range(total_coloum - 3):
        for row in range(total_row - 3):
            if (board[row][coloum] == piece and
                board[row + 1][coloum + 1] == piece and
                board[row + 2][coloum + 2] == piece and
                board[row + 3][coloum + 3] == piece):
                return True

    # diagonal negative
    for coloum in range(total_coloum - 3):
        for row in range(3, total_row):
            if (board[row][coloum] == piece and
                board[row - 1][coloum + 1] == piece and
                board[row - 2][coloum + 2] == piece and
                board[row - 3][coloum + 3] == piece):
                return True

    return False

def draw_board(board, screen, squre_size, radius, height):
    # board background + holes
    for c in range(total_coloum):
        for r in range(total_row):
            pygame.draw.rect(
                screen, BLUE,
                (c * squre_size, (r + 1) * squre_size, squre_size, squre_size)
            )
            pygame.draw.circle(
                screen, BLACK,
                (c * squre_size + squre_size // 2, (r + 1) * squre_size + squre_size // 2),
                radius
            )

    # pieces (remember: r=0 is bottom in board, but y grows downward in screen)
    for c in range(total_coloum):
        for r in range(total_row):
            if board[r][c] == 1:
                pygame.draw.circle(
                    screen, RED,
                    (c * squre_size + squre_size // 2,
                     height - (r * squre_size + squre_size // 2)),
                    radius
                )
            elif board[r][c] == 2:
                pygame.draw.circle(
                    screen, YELLOW,
                    (c * squre_size + squre_size // 2,
                     height - (r * squre_size + squre_size // 2)),
                    radius
                )

    pygame.display.update()

def main():
    board = create_board()
    game_over = False
    turn = 0  # 0 -> player 1 (RED), 1 -> player 2 (YELLOW)

    pygame.init()

    squre_size = 100
    width = total_coloum * squre_size
    height = (total_row + 1) * squre_size  # +1 top area for hover piece
    radius = squre_size // 2 - 8

    screen = pygame.display.set_mode((width, height))
    pygame.display.set_caption("Connect-4 GUI (Pygame)")

    font = pygame.font.SysFont("Arial", 52, bold=True)

    draw_board(board, screen, squre_size, radius, height)

    while True:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()

            # mouse move -> show hover piece
            if event.type == pygame.MOUSEMOTION and not game_over:
                pygame.draw.rect(screen, BLACK, (0, 0, width, squre_size))
                x = event.pos[0]
                if turn == 0:
                    pygame.draw.circle(screen, RED, (x, squre_size // 2), radius)
                else:
                    pygame.draw.circle(screen, YELLOW, (x, squre_size // 2), radius)
                pygame.display.update()

            # click -> drop piece
            if event.type == pygame.MOUSEBUTTONDOWN and not game_over:
                pygame.draw.rect(screen, BLACK, (0, 0, width, squre_size))

                x = event.pos[0]
                coloum = x // squre_size

                if 0 <= coloum < total_coloum and is_vlid_location(board, coloum):
                    row = get_next_open_row(board, coloum)
                    if row is None:
                        continue

                    piece = 1 if turn == 0 else 2
                    drop_piece(board, row, coloum, piece)

                    if winning_move(board, piece):
                        label = font.render(
                            "Player 1 Wins!" if piece == 1 else "Player 2 Wins!",
                            True,
                            RED if piece == 1 else YELLOW
                        )
                        screen.blit(label, (20, 10))
                        game_over = True

                    draw_board(board, screen, squre_size, radius, height)

                    # switch turn
                    turn ^= 1

                # optional: tie check
                if not game_over and np.all(board != 0):
                    label = font.render("TIE!", True, (200, 200, 200))
                    screen.blit(label, (20, 10))
                    game_over = True
                    pygame.display.update()

        if game_over:
            # small delay so user can see result
            pygame.time.wait(2500)
            pygame.quit()
            sys.exit()

if __name__ == "__main__":
    main()
